# 01 — Data and split check

**Purpose.** Prove that the upload arrived complete and unchanged (SHA-256 of all 400 videos and every code file), that the split is leakage-free, and show the train / val / test tables and charts.

**Kaggle settings.** Accelerator **None** (CPU is enough) · Internet Off is fine.

**Inputs.** The private dataset `bdsl-k400`.

**Outputs** (under `/kaggle/working/k400/`, saved by *Save Version → Save & Run All*). `outputs/data_check/` (verification output saved in this notebook).

**Expected run time** [ESTIMATE, not measured yet]. about 3–8 minutes (hashing about 5.9 GB).

**What to check.** `UPLOAD CHECK: PASS`, `RESULT: PASS (10 passed, 0 failed, 0 skipped)`, counts 320 / 40 / 40.

> Honesty rules: label every number measured or estimated; results are small-scale and signer-dependent;
> the test split is used once (notebook 07); low coordinate error is not proof of correct sign language;
> G0 glosses are pseudo-glosses, not BdSL annotation; no human or expert validation exists.

In [ ]:
# Cell 1 - Setup (the same in every notebook). Finds the data and the code, sets the paths, and copies the
# outputs of earlier notebooks (added with "Add Input") into /kaggle/working/k400 so results accumulate.
import os, sys, zipfile, pathlib, json
NOTEBOOK = "01_data_split_check"

def _find_root():
    base = pathlib.Path("/kaggle/input")
    if base.is_dir():
        for dirpath, dirnames, filenames in os.walk(base):
            if "UPLOAD_MANIFEST.csv" in filenames and "PACKAGE_INFO.json" in filenames:
                return pathlib.Path(dirpath)
            depth = len(pathlib.Path(dirpath).relative_to(base).parts)
            dirnames[:] = [] if depth >= 4 else [d for d in dirnames if d not in ("videos", "k400")]
        zips = sorted(base.rglob("bdsl-k400-*.zip"))
        if zips:  # Kaggle kept the upload as one zip file: unpack it once (a few minutes)
            dest = pathlib.Path("/tmp/k400_dataset")   # /tmp is not saved as Output
            if not (dest / "UPLOAD_MANIFEST.csv").is_file():
                print("Unpacking", zips[0], "->", dest)
                zipfile.ZipFile(zips[0]).extractall(dest)
            return dest
        raise FileNotFoundError("The private dataset is not attached. Right panel: Add Input -> Your Datasets -> bdsl-k400.")
    here = pathlib.Path.cwd().resolve()
    for c in [here, *here.parents]:
        if (c / "pyproject.toml").is_file() and (c / "kaggle" / "src" / "k400").is_dir():
            return c
    raise FileNotFoundError("Open this notebook inside the project folder, or on Kaggle with the dataset attached.")

DATASET_ROOT = _find_root()
CODE_ROOT = DATASET_ROOT / "code" if (DATASET_ROOT / "code" / "pyproject.toml").is_file() else DATASET_ROOT
for _p in (CODE_ROOT / "kaggle" / "src", CODE_ROOT / "src"):
    sys.path.insert(0, str(_p))
from k400.setup import setup
S = setup(DATASET_ROOT, CODE_ROOT, NOTEBOOK)

## Step 1 — Every uploaded file against `UPLOAD_MANIFEST.csv`

In [ ]:
rc_upload = S.k400("verify-upload", "--root", str(DATASET_ROOT))

## Step 2 — Leakage and consistency tests (the same as `kaggle/src/verify_split.py`)

In [ ]:
rc_split = S.k400("verify", "--require-videos")

## Step 3 — Tables per split (from the manifests)

In [ ]:
import pandas as pd
from IPython.display import Image, Markdown, display
sub = pd.read_csv(DATASET_ROOT / "data/subset_manifest.csv", encoding="utf-8-sig")
display(sub.groupby("split").agg(clips=("index", "size"), groups=("group_id", "nunique"), mean_duration_s=("duration_s", "mean"), minimal_pair_heldout=("mp_role", lambda s: (s == "heldout").sum())))
for col in ["length_bucket", "question", "negation", "orientation", "resolution_bucket", "duration_quartile"]:
    display(pd.crosstab(sub[col], sub["split"], margins=True))
meta = json.load(open(DATASET_ROOT / "data/SELECTION_META.json", encoding="utf-8"))
display(pd.DataFrame(meta["oov"]).drop(columns="train_vocabulary_size", errors="ignore"))
print("train vocabulary:", meta["oov"]["train_vocabulary_size"], "word types")

## Step 4 — Chart and the split report

In [ ]:
display(Image(filename=str(DATASET_ROOT / "data/SPLIT_SUMMARY.png")))
display(Markdown((DATASET_ROOT / "data/SPLIT_REPORT.md").read_text(encoding="utf-8")))

In [ ]:
(S.state_root / "outputs/data_check").mkdir(parents=True, exist_ok=True)
json.dump({"upload_check_exit": rc_upload, "split_check_exit": rc_split}, open(S.state_root / "outputs/data_check/result.json", "w"))
S.finish(checks=[
    ("every uploaded file identical (SHA-256)", rc_upload == 0),
    ("all 10 split checks pass", rc_split == 0),
    ("counts 320/40/40", sub["split"].value_counts().to_dict() == {"train": 320, "val": 40, "test": 40}),
])